# IOAI 2025 NLP Test – Cross-Lingual Affect in Tweets

In this challenge, students are asked to develop a system for zero-shot affect classification across languages. The system must be trained using English labeled data and then deployed on a target language (Spanish) where only unlabeled test data is available.

The task is formulated as a binary classification problem. The input is a tweet, and the goal is to predict the overall affect conveyed. In the following, you are provided with a code base containing data processing, basic fine-tuning over a multilingual base language model, and evaluation.

You are also provided with the following datasets:
The labeled training and validation data (DE_train and DE_valid) comes from

*   labeled training data in English (DE_train)
*   labeled validation data in English (DE_valid)
*   unlabeled corpus data in Spanish (DT_unlabeled)
*   unlabeled test data in Spanish (DT_test_unlabeled)

Your goal is to predict labels in the test set of the target language (DT_test_unlabeled).

# 🔧 Install Required Libraries

This cell installs the necessary libraries, including datasets (for loading Hugging Face datasets).

In [ ]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12

# 📁 Clone Data Repository

This cell clones a GitHub repo that contain training and testing datasets.

In [ ]:
!git clone https://github.com/duyngtr16061999/Cross_lingual_sentiment_classification

Cloning into 'Cross_lingual_sentiment_classification'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 18 (delta 9), reused 17 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 782.28 KiB | 11.85 MiB/s, done.
Resolving deltas: 100% (9/9), done.


# 📦 Load & Preprocess Data

This section loads the training data in English (DE_train.json), Spanish corpus (DT_unlabeled.json) and test data in Spanish (DT_test_unlabeled.json). Each text is tokenized using xlm-roberta-base tokenizer and padded/truncated for model input.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import torch

In [ ]:
# Load datasets
de_train = load_dataset("json", data_files="Cross_lingual_sentiment_classification/DE_train.json", split="train")
de_valid = load_dataset("json", data_files="Cross_lingual_sentiment_classification/DE_valid.json", split="train")
dt_test = load_dataset("json", data_files="Cross_lingual_sentiment_classification/DT_test_unlabeled.json", split="train")
dt = load_dataset("json", data_files="Cross_lingual_sentiment_classification/DT_unlabeled.json", split="train")
# Load tokenizer and model
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=32)

def preprocess_dataset(dataset):
  # Encode the input data
  dataset = dataset.map(tokenize_function, batched=True)
  # The transformers model expects the target class column to be named "labels"
  dataset = dataset.rename_column("label", "labels")
  # Transform to pytorch tensors and only output the required columns
  dataset.set_format(type="torch",columns=["input_ids", "attention_mask", "labels"])
  return dataset

def preprocess_dataset_unlabeled(dataset):
  # Encode the input data
  dataset = dataset.map(tokenize_function, batched=True)
  # Transform to pytorch tensors and only output the required columns
  dataset.set_format(type="torch",columns=["input_ids", "attention_mask"])
  return dataset

tokenized_train = preprocess_dataset(de_train)
tokenized_valid = preprocess_dataset(de_valid)
tokenized_test = preprocess_dataset_unlabeled(dt_test)

Map:   0%|          | 0/6617 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/2687 [00:00<?, ? examples/s]

In [ ]:
de_train

Dataset({
    features: ['text', 'label'],
    num_rows: 6617
})

In [ ]:
dt_test

Dataset({
    features: ['text'],
    num_rows: 2687
})

In [ ]:
dt

Dataset({
    features: ['text'],
    num_rows: 3357
})

# 🤖 Load Model

Here, we load a multilingual base model `xlm-roberta-base`. You are required to only use `xlm-roberta-base` as the backbone LM to solve this problem. Other pre-trained LLMs are not allowed.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model_name = "xlm-roberta-base"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# 🏋️‍♂️ Train Model on English Data

This block configures the training process using Trainer from Hugging Face. It fine-tunes the model on the labeled English training set for 2 epochs.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    learning_rate=1e-4,
    num_train_epochs=2,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    logging_steps=50,
    output_dir="./training_output",
    overwrite_output_dir=True,
    # The next line is important to ensure the dataset labels are properly passed to the model
    remove_unused_columns=False,
    report_to="none",

)

train_dataset = tokenized_train

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

In [ ]:
trainer.train()

Step,Training Loss


# 🧪 Evaluate Model on Validation Set

This block defines an evaluation function to compute accuracy and uses Hugging Face’s Trainer class to evaluate the model on a labeled validation set. The compute_accuracy function calculates the percentage of correct predictions. The Trainer is initialized in evaluation-only mode (no training), and the final accuracy is printed using .evaluate().

Note: WANDB_DISABLED is set to true to prevent Weights & Biases logging in Colab.

In [ ]:
import numpy as np
from transformers import EvalPrediction
import os
os.environ["WANDB_DISABLED"] = "true"

def compute_accuracy(p: EvalPrediction):
  preds = np.argmax(p.predictions, axis=1)
  return {"acc": (preds == p.label_ids).mean()}
eval_trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir="./eval_output", remove_unused_columns=False,),
    eval_dataset=tokenized_valid,
    compute_metrics=compute_accuracy,
)
eval_trainer.evaluate()

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


{'eval_loss': 0.6898102164268494,
 'eval_model_preparation_time': 0.0028,
 'eval_acc': 0.5607798165137615,
 'eval_runtime': 1.7999,
 'eval_samples_per_second': 484.465,
 'eval_steps_per_second': 60.558}

# 🔍 Predict on Spanish Test Data

In this section, the model makes predictions on the Spanish test set (DT_test_unlabeled) one sample at a time (to avoid batch-related errors). Predictions are stored in a CSV file.

In [ ]:
model.eval()
preds = []

with torch.no_grad():
    for sample in tokenized_test:
        input_ids = sample["input_ids"].unsqueeze(0).to(model.device)
        attention_mask = sample["attention_mask"].unsqueeze(0).to(model.device)

        output = model(input_ids=input_ids, attention_mask=attention_mask)
        pred = torch.argmax(output.logits, dim=-1).item()
        preds.append(pred)

In [ ]:
import pandas as pd

df_preds = pd.DataFrame({
    "text": dt_test["text"],
    "predicted_label": preds
})
df_preds.to_csv("DT_test_predictions.csv", index=False)

# 👉 Your Task

You are tasked to solve this cross-lingual transfer learning task - utilising the labeled English training data and the unlabeled Spanish corpus to enhance the sentiment prediction performance on the Spanish test dataset. Your solution should satisfy the following requirements:



*   You can **only** use `xlm-roberta-base` as the backbone language model.
*   Translation tools or models are **not allowed**.
*   Other pre-trained LLMs (including ChatGPT) are **not allowed**.
*   You are **not allowed** to refer to your existing notebooks.
*   You are encouraged to come up with efficient solutions which demand less computational resource.

Assessment will be based on innovation, performance and efficiency.






# ✅  Submission

You are required to submit

1.   a prediction file in csv format using the above prediction script.
2.   your solution code in .ipynb



Name your submission files in the format of "Yourname_NLP_prediction.csv" and "Yourname_NLP_code.ipynb".

The submission link is: https://forms.gle/aZmBJ26fvjBW79H57